# S&P 500 Momentum Strategy — Full Quant Backtest

**Signal:** 12-1 month cross-sectional momentum  
**Universe:** Top 50 S&P 500 stocks by market cap  
**Portfolio:** Top 20 momentum winners, MVO-sized (Ledoit-Wolf + Black-Litterman)  
**Rebalance:** Monthly  
**Risk:** Per-position stop-loss, max drawdown circuit breaker  
**Execution:** Bid-ask spread + market impact slippage  
**Benchmark:** SPY (buy-and-hold)

In [ ]:
# ── Auto-install all dependencies ─────────────────────────────────────────
import subprocess, sys

REQUIRED = [
    'yfinance',
    'pandas',
    'numpy',
    'scipy',
    'scikit-learn',
    'matplotlib',
]

for pkg in REQUIRED:
    try:
        __import__(pkg.replace('-', '_').split('[')[0])
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ↓ installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  ✓ {pkg} installed')

print('\nAll dependencies ready.')

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.optimize as opt
from sklearn.covariance import LedoitWolf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
print('Imports OK')

## 1 · Universe & Parameters

In [ ]:
# ── Universe: top 50 S&P 500 by market cap (as of 2024) ───────────────────
UNIVERSE = [
    'AAPL','MSFT','NVDA','AMZN','GOOGL','META','TSLA','BRK-B','JPM','LLY',
    'V','UNH','XOM','AVGO','MA','PG','JNJ','COST','HD','MRK',
    'ABBV','CVX','CRM','BAC','NFLX','AMD','PEP','KO','TMO','ACN',
    'MCD','CSCO','ABT','WMT','LIN','DHR','TXN','NEE','PM','QCOM',
    'RTX','HON','INTU','AMGN','IBM','GE','CAT','SPGI','LOW','AMAT'
]

# ── Strategy parameters ───────────────────────────────────────────────────
START          = '2015-01-01'
END            = '2025-01-01'
LOOKBACK_LONG  = 252        # 12 months in trading days
LOOKBACK_SHORT = 21         # 1 month skip (12-1 momentum)
N_HOLDINGS     = 20         # top momentum stocks to hold
W_MAX          = 0.15       # max weight per stock
RF_ANNUAL      = 0.04       # risk-free rate
RF_DAILY       = RF_ANNUAL / 252

# ── Execution costs ───────────────────────────────────────────────────────
BID_ASK_BPS    = 5          # bid-ask spread cost in basis points (one-way)
MARKET_IMPACT  = 0.001      # 10bps market impact per trade (one-way)

# ── Risk management ───────────────────────────────────────────────────────
STOP_LOSS      = -0.12      # exit position if down >12% from entry
MAX_DD_HALT    = -0.20      # halt all trading if portfolio DD > 20%

print(f'Universe   : {len(UNIVERSE)} stocks')
print(f'Period     : {START} to {END}')
print(f'Holdings   : top {N_HOLDINGS} momentum stocks')
print(f'Max weight : {W_MAX:.0%} per stock')
print(f'Exec cost  : {BID_ASK_BPS}bps bid-ask + {MARKET_IMPACT*100:.0f}bps market impact')
print(f'Stop-loss  : {STOP_LOSS:.0%} from entry price')
print(f'DD halt    : {MAX_DD_HALT:.0%} portfolio drawdown')

## 2 · Data Download

In [ ]:
print('Downloading price data (this takes ~30 seconds)...')
raw = yf.download(
    UNIVERSE + ['SPY'],
    start=START,
    end=END,
    auto_adjust=True,
    progress=True
)['Close']

# Drop tickers with >5% missing data, forward-fill remaining gaps
missing = raw.isnull().mean()
dropped = missing[missing >= 0.05].index.tolist()
if dropped:
    print(f'Dropped (too many gaps): {dropped}')
raw = raw.loc[:, missing < 0.05].ffill()

# Separate benchmark
spy     = raw['SPY'].copy()
prices  = raw.drop(columns=['SPY'])
returns = prices.pct_change()
spy_ret = spy.pct_change()

print(f'\nLoaded {prices.shape[1]} stocks x {prices.shape[0]} days')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')

## 3 · Signal: 12-1 Month Cross-Sectional Momentum

The **12-1 momentum** signal (Jegadeesh & Titman, 1993) measures cumulative return from 12 months ago to 1 month ago, skipping the most recent month to avoid short-term reversal. At each rebalance date we rank all stocks by this signal and select the top 20.

In [ ]:
def compute_momentum(prices_df, long_lb=LOOKBACK_LONG, short_lb=LOOKBACK_SHORT):
    """12-1 momentum: return from t-252 to t-21."""
    return prices_df.shift(short_lb) / prices_df.shift(long_lb) - 1

momentum = compute_momentum(prices)

# Last trading day of each month, after we have enough history
monthly_dates = prices.resample('ME').last().index
monthly_dates = monthly_dates[monthly_dates >= prices.index[LOOKBACK_LONG + 10]]

print(f'Rebalance dates: {len(monthly_dates)} months')
print(f'First rebalance: {monthly_dates[0].date()}')
print(f'Last rebalance : {monthly_dates[-1].date()}')

## 4 · Portfolio Construction: MVO with Ledoit-Wolf + Black-Litterman

Each month:
1. Rank stocks by 12-1 momentum, select top 20
2. Estimate covariance on trailing 252-day returns using **Ledoit-Wolf shrinkage**
3. Derive expected returns via **Black-Litterman** equilibrium, tilted by momentum z-scores
4. Solve constrained MVO for the **maximum Sharpe** portfolio

In [ ]:
def get_bl_returns(ret_window, w_eq, delta=2.5):
    """Black-Litterman equilibrium returns anchored to equal-weight market proxy."""
    lw = LedoitWolf().fit(ret_window)
    Sigma = lw.covariance_ * 252
    Pi = delta * Sigma @ w_eq
    return Pi, Sigma


def max_sharpe(mu, Sigma, rf=RF_ANNUAL, w_max=W_MAX):
    """Maximise Sharpe ratio subject to long-only + weight cap."""
    n = len(mu)
    def neg_sharpe(w):
        r   = mu @ w
        vol = np.sqrt(w @ Sigma @ w)
        return -(r - rf) / vol if vol > 1e-8 else 0.0
    res = opt.minimize(
        neg_sharpe, np.ones(n) / n,
        method='SLSQP',
        bounds=[(0.0, w_max)] * n,
        constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
        options={'maxiter': 500, 'ftol': 1e-9}
    )
    return res.x if res.success else np.ones(n) / n


def select_and_size(date, prices_df, returns_df, momentum_df, n_hold=N_HOLDINGS):
    """At a rebalance date: rank by momentum, select top-N, size via MVO."""
    mom_today = momentum_df.loc[date].dropna()
    if len(mom_today) < n_hold:
        return {}

    selected = mom_today.nlargest(n_hold).index.tolist()

    ret_window = returns_df.loc[:date, selected].tail(252).dropna()
    if ret_window.shape[0] < 60:
        return {t: 1 / n_hold for t in selected}

    w_eq = np.ones(n_hold) / n_hold
    Pi, Sigma = get_bl_returns(ret_window.values, w_eq)

    # Tilt BL returns toward higher-momentum stocks via z-score
    mom_scores = mom_today[selected].values
    mom_z = (mom_scores - mom_scores.mean()) / (mom_scores.std() + 1e-8)
    mu_bl = Pi + 0.02 * mom_z

    weights = max_sharpe(mu_bl, Sigma)
    return dict(zip(selected, weights))


print('Portfolio construction functions ready.')
print('Testing on first rebalance date...')
test = select_and_size(monthly_dates[0], prices, returns, momentum)
if test:
    print(f'Selected {len(test)} stocks, weights sum = {sum(test.values()):.4f}')
    print('Top 5 positions:')
    for t, w in sorted(test.items(), key=lambda x: -x[1])[:5]:
        print(f'  {t:8s} {w:.2%}')

## 5 · Backtest Engine

Simulates daily P&L with:
- **Monthly rebalancing** with bid-ask + market impact costs
- **Per-position stop-loss**: exit if stock falls >12% from entry price
- **Portfolio circuit breaker**: halt trading if drawdown exceeds 20%, resume at -10%

In [ ]:
def compute_transaction_cost(old_weights, new_weights):
    """Turnover-scaled bid-ask + market impact cost."""
    all_tickers = set(old_weights) | set(new_weights)
    turnover = sum(
        abs(new_weights.get(t, 0.0) - old_weights.get(t, 0.0))
        for t in all_tickers
    )
    cost_per_unit = BID_ASK_BPS / 10000 + MARKET_IMPACT
    return turnover * cost_per_unit


def run_backtest(prices_df, returns_df, momentum_df, spy_returns):
    trading_days = prices_df.index
    rebal_set    = set(monthly_dates)

    portfolio_val   = [1.0]
    daily_ret       = []
    current_weights = {}
    entry_prices    = {}
    halted          = False
    peak_val        = 1.0
    turnover_log    = []
    holdings_log    = []

    for date in trading_days[1:]:
        pv       = portfolio_val[-1]
        peak_val = max(peak_val, pv)
        dd       = (pv - peak_val) / peak_val

        # ── Circuit breaker ───────────────────────────────────────────────
        if dd < MAX_DD_HALT and not halted:
            print(f'  Circuit breaker ON  {date.date()}  (DD={dd:.1%})')
            halted = True
        if halted and dd > -0.10:
            print(f'  Circuit breaker OFF {date.date()}')
            halted = False

        # ── Per-position stop-loss ─────────────────────────────────────────
        if not halted and entry_prices:
            stopped = [
                t for t, ep in list(entry_prices.items())
                if t in prices_df.columns
                and (prices_df.loc[date, t] / ep - 1) < STOP_LOSS
            ]
            for t in stopped:
                current_weights.pop(t, None)
                entry_prices.pop(t, None)
            if current_weights:
                total = sum(current_weights.values())
                if total > 0:
                    current_weights = {t: w / total for t, w in current_weights.items()}

        # ── Monthly rebalance ─────────────────────────────────────────────
        if date in rebal_set and not halted:
            new_w = select_and_size(date, prices_df, returns_df, momentum_df)
            if new_w:
                tc = compute_transaction_cost(current_weights, new_w)
                tv = sum(
                    abs(new_w.get(t, 0) - current_weights.get(t, 0))
                    for t in set(current_weights) | set(new_w)
                )
                turnover_log.append({'date': date, 'turnover': tv, 'tc': tc})
                current_weights = new_w
                for t in current_weights:
                    if t in prices_df.columns:
                        entry_prices[t] = prices_df.loc[date, t]
                pv *= (1 - tc)

        # ── Daily P&L ─────────────────────────────────────────────────────
        if halted or not current_weights:
            dr = RF_DAILY
        else:
            dr = sum(
                w * returns_df.loc[date, t]
                for t, w in current_weights.items()
                if t in returns_df.columns
            )

        daily_ret.append(dr)
        portfolio_val.append(pv * (1 + dr))
        holdings_log.append({
            'date': date,
            'n_holdings': len(current_weights),
            'halted': halted
        })

    dates       = trading_days[1:]
    port_series = pd.Series(portfolio_val[1:], index=dates)
    ret_series  = pd.Series(daily_ret, index=dates)
    spy_cum     = (1 + spy_returns.reindex(dates).fillna(0)).cumprod()
    turnover_df = pd.DataFrame(turnover_log).set_index('date') if turnover_log else pd.DataFrame()
    holdings_df = pd.DataFrame(holdings_log).set_index('date')

    return port_series, ret_series, spy_cum, turnover_df, holdings_df


print('Running backtest...')
port_series, ret_series, spy_cum, turnover_df, holdings_df = run_backtest(
    prices, returns, momentum, spy_ret
)
print(f'Done. {len(ret_series)} trading days simulated.')
print(f'Final portfolio value: ${port_series.iloc[-1]:.2f}')

## 6 · Performance Tearsheet

In [ ]:
def compute_metrics(ret_series, periods=252):
    r        = ret_series.dropna()
    ann_ret  = (1 + r).prod() ** (periods / len(r)) - 1
    ann_vol  = r.std() * np.sqrt(periods)
    sharpe   = (ann_ret - RF_ANNUAL) / ann_vol
    downside = r[r < RF_DAILY].std() * np.sqrt(periods)
    sortino  = (ann_ret - RF_ANNUAL) / downside if downside > 0 else np.nan
    cum      = (1 + r).cumprod()
    max_dd   = ((cum - cum.cummax()) / cum.cummax()).min()
    calmar   = ann_ret / abs(max_dd) if max_dd != 0 else np.nan
    win_rate = (r > 0).mean()
    var_95   = np.percentile(r, 5)
    cvar_95  = r[r <= var_95].mean()
    return {
        'Ann. Return':     ann_ret,
        'Ann. Volatility': ann_vol,
        'Sharpe Ratio':    sharpe,
        'Sortino Ratio':   sortino,
        'Calmar Ratio':    calmar,
        'Max Drawdown':    max_dd,
        'Win Rate':        win_rate,
        'Best Day':        r.max(),
        'Worst Day':       r.min(),
        'VaR 95% (daily)': var_95,
        'CVaR 95% (daily)':cvar_95,
    }


spy_ret_aligned = spy_ret.reindex(ret_series.index).fillna(0)
strat_metrics   = compute_metrics(ret_series)
spy_metrics     = compute_metrics(spy_ret_aligned)

pct_keys = {
    'Ann. Return','Ann. Volatility','Max Drawdown',
    'Win Rate','Best Day','Worst Day','VaR 95% (daily)','CVaR 95% (daily)'
}
rows = []
for k in strat_metrics:
    fmt = '{:.2%}' if k in pct_keys else '{:.2f}'
    rows.append({
        'Metric':   k,
        'Strategy': fmt.format(strat_metrics[k]),
        'SPY B&H':  fmt.format(spy_metrics[k]),
    })

summary = pd.DataFrame(rows).set_index('Metric')
print('=' * 50)
print('  PERFORMANCE TEARSHEET')
print('=' * 50)
print(summary.to_string())

if not turnover_df.empty:
    print(f"\nAvg monthly turnover : {turnover_df['turnover'].mean():.1%}")
    print(f"Avg transaction cost : {turnover_df['tc'].mean()*100:.2f} bps/rebalance")
    print(f"Total cost drag      : {turnover_df['tc'].sum()*100:.0f} bps over full period")

## 7 · Tearsheet Plots

In [ ]:
fig = plt.figure(figsize=(14, 18))
gs  = gridspec.GridSpec(5, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. Cumulative return ───────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(port_series.index, port_series.values,
         color='#1565C0', lw=1.8, label='Momentum Strategy')
ax1.plot(spy_cum.index, spy_cum.values,
         color='#757575', lw=1.2, linestyle='--', label='SPY Buy & Hold')
ax1.set_yscale('log')
ax1.set_title('Cumulative Return (log scale)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Growth of $1')
ax1.legend(fontsize=9)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y:.1f}'))

# ── 2. Drawdown ───────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, :])
dd_strat = port_series / port_series.cummax() - 1
dd_spy   = spy_cum / spy_cum.cummax() - 1
ax2.fill_between(dd_strat.index, dd_strat.values, 0,
                 alpha=0.4, color='#1565C0', label='Strategy DD')
ax2.plot(dd_spy.index, dd_spy.values,
         color='#757575', lw=0.8, linestyle='--', label='SPY DD')
ax2.set_title('Drawdown', fontsize=12, fontweight='bold')
ax2.set_ylabel('Drawdown')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax2.legend(fontsize=9)

# ── 3. Rolling 12M Sharpe ─────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2, 0])
roll_sharpe = (
    ret_series.rolling(252).mean() * 252 - RF_ANNUAL
) / (ret_series.rolling(252).std() * np.sqrt(252))
ax3.plot(roll_sharpe.index, roll_sharpe.values, color='#1565C0', lw=1.2)
ax3.axhline(0, color='black', lw=0.8)
ax3.axhline(1, color='green', lw=0.6, linestyle=':', alpha=0.7, label='SR=1')
ax3.set_title('Rolling 12M Sharpe Ratio', fontsize=11, fontweight='bold')
ax3.set_ylabel('Sharpe')
ax3.legend(fontsize=8)

# ── 4. Rolling 12M Volatility ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 1])
ax4.plot(ret_series.rolling(252).std() * np.sqrt(252),
         color='#1565C0', lw=1.2, label='Strategy')
ax4.plot(spy_ret_aligned.rolling(252).std() * np.sqrt(252),
         color='#757575', lw=1.0, linestyle='--', label='SPY')
ax4.set_title('Rolling 12M Volatility', fontsize=11, fontweight='bold')
ax4.set_ylabel('Annualised Vol')
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax4.legend(fontsize=8)

# ── 5. Return distribution ────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[3, 0])
ax5.hist(ret_series.dropna() * 100, bins=80,
         color='#1565C0', alpha=0.7, edgecolor='white', lw=0.3, label='Strategy')
ax5.hist(spy_ret_aligned.dropna() * 100, bins=80,
         color='#757575', alpha=0.4, edgecolor='white', lw=0.3, label='SPY')
ax5.axvline(0, color='black', lw=0.8)
ax5.set_title('Daily Return Distribution', fontsize=11, fontweight='bold')
ax5.set_xlabel('Daily Return (%)')
ax5.legend(fontsize=8)

# ── 6. Monthly returns heatmap ────────────────────────────────────────────
ax6 = fig.add_subplot(gs[3, 1])
monthly_ret = ret_series.resample('ME').apply(lambda x: (1 + x).prod() - 1)
monthly_ret.index = monthly_ret.index.to_period('M')
pivot = monthly_ret.groupby(
    [monthly_ret.index.year, monthly_ret.index.month]
).first().unstack()
im = ax6.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-0.15, vmax=0.15)
ax6.set_yticks(range(len(pivot.index)))
ax6.set_yticklabels(pivot.index, fontsize=7)
ax6.set_xticks(range(12))
ax6.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'], fontsize=7)
ax6.set_title('Monthly Returns Heatmap', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax6, format='{x:.0%}', shrink=0.8)

# ── 7. Monthly turnover ───────────────────────────────────────────────────
ax7 = fig.add_subplot(gs[4, 0])
if not turnover_df.empty:
    ax7.bar(turnover_df.index, turnover_df['turnover'] * 100,
            color='#1565C0', alpha=0.7, width=20)
    ax7.axhline(
        turnover_df['turnover'].mean() * 100,
        color='red', lw=1, linestyle='--',
        label=f"Avg {turnover_df['turnover'].mean():.0%}"
    )
    ax7.set_title('Monthly Portfolio Turnover', fontsize=11, fontweight='bold')
    ax7.set_ylabel('Turnover (%)')
    ax7.legend(fontsize=8)

# ── 8. Number of holdings ─────────────────────────────────────────────────
ax8 = fig.add_subplot(gs[4, 1])
ax8.plot(holdings_df.index, holdings_df['n_holdings'],
         color='#1565C0', lw=0.8)
ax8.fill_between(holdings_df.index, holdings_df['n_holdings'],
                 alpha=0.2, color='#1565C0')
halted_days = holdings_df[holdings_df['halted']]
if not halted_days.empty:
    ax8.scatter(halted_days.index, [0] * len(halted_days),
                color='red', s=2, label='Halted (circuit breaker)', zorder=3)
    ax8.legend(fontsize=8)
ax8.set_title('Number of Holdings', fontsize=11, fontweight='bold')
ax8.set_ylabel('# Stocks')
ax8.set_ylim(0, N_HOLDINGS + 2)

fig.suptitle(
    'S&P 500 Momentum Strategy — Full Backtest Tearsheet\n'
    f'12-1 Month Momentum · Top {N_HOLDINGS} Stocks · Monthly Rebalance · MVO Sizing',
    fontsize=13, fontweight='bold', y=0.98
)
plt.savefig('tearsheet.png', dpi=130, bbox_inches='tight')
plt.show()
print('Tearsheet saved to tearsheet.png')

## 8 · Annual Returns Summary

In [ ]:
annual_strat = ret_series.resample('YE').apply(lambda x: (1 + x).prod() - 1)
annual_spy   = spy_ret_aligned.resample('YE').apply(lambda x: (1 + x).prod() - 1)
annual_df    = pd.DataFrame({
    'Strategy': annual_strat,
    'SPY B&H':  annual_spy,
    'Alpha':    annual_strat - annual_spy,
})
annual_df.index = annual_df.index.year
annual_df.index.name = 'Year'

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(annual_df))
w = 0.35
ax.bar(x - w/2, annual_df['Strategy'] * 100, w,
       label='Momentum Strategy', color='#1565C0', alpha=0.85)
ax.bar(x + w/2, annual_df['SPY B&H'] * 100, w,
       label='SPY Buy & Hold', color='#757575', alpha=0.7)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(annual_df.index)
ax.set_title('Annual Returns: Strategy vs SPY', fontsize=12, fontweight='bold')
ax.set_ylabel('Return (%)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(annual_df.apply(lambda col: col.map('{:.1%}'.format)).to_string())